In [0]:
# Step 1: Setup database/schema

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA census360")

print("Using schema: workspace.census360")

In [0]:
# Step 2: Check available Silver tables

spark.sql("""
SHOW TABLES IN workspace.census360
""").show(truncate=False)

In [0]:
# Step 3: Load main Silver GN demographic base table

silver_gn_base = spark.table("workspace.census360.silver_gn_demographic_base")

display(silver_gn_base.limit(10))

In [0]:
# Step 4: Validate Silver GN base table

silver_gn_base.selectExpr(
    "count(*) as total_rows",
    "count(distinct gnd_uid) as distinct_gnd_uid",
    "count(distinct district_name) as district_count",
    "count(distinct ds_division_name) as ds_division_count"
).show()

In [0]:
from pyspark.sql.functions import col, when, round, greatest, least

In [0]:
# Create GN-level Gold telco features from Silver demographic base

gold_gn_telco_features = silver_gn_base.select(
    "gnd_uid",
    "province_code",
    "province_name",
    "district_code",
    "district_name",
    "ds_division_code",
    "ds_division_name",
    "gn_division_code",
    "gn_division_number",
    "gn_division_name",
    "local_government_code",
    "local_government_name",
    "population_total",
    "male_population",
    "female_population",
    "age_total",
    "age_0_14",
    "age_15_59",
    "age_60_64",
    "age_65_plus",
    "occupied_housing_units"
).withColumn(
    "household_size",
    round(
        when(col("occupied_housing_units") > 0,
             col("population_total") / col("occupied_housing_units")
        ).otherwise(None),
        2
    )
).withColumn(
    "male_share",
    round(
        when(col("population_total") > 0,
             col("male_population") / col("population_total")
        ).otherwise(None),
        4
    )
).withColumn(
    "female_share",
    round(
        when(col("population_total") > 0,
             col("female_population") / col("population_total")
        ).otherwise(None),
        4
    )
).withColumn(
    "child_share",
    round(
        when(col("population_total") > 0,
             col("age_0_14") / col("population_total")
        ).otherwise(None),
        4
    )
).withColumn(
    "working_age_share",
    round(
        when(col("population_total") > 0,
             col("age_15_59") / col("population_total")
        ).otherwise(None),
        4
    )
).withColumn(
    "elderly_share",
    round(
        when(col("population_total") > 0,
             (col("age_60_64") + col("age_65_plus")) / col("population_total")
        ).otherwise(None),
        4
    )
)

In [0]:
display(gold_gn_telco_features.limit(10))

In [0]:
gold_gn_telco_features.selectExpr(
    "count(*) as total_rows",
    "count(distinct gnd_uid) as distinct_gnd_uid",
    "count(distinct district_name) as district_count",
    "count(distinct ds_division_name) as ds_division_count"
).show()

In [0]:
gold_gn_telco_features.write.mode("overwrite").saveAsTable(
    "workspace.census360.gold_gn_telco_features"
)

print("gold_gn_telco_features created successfully.")

In [0]:
spark.table("workspace.census360.gold_gn_telco_features").count()

In [0]:
from pyspark.sql.functions import col, when, round, min as spark_min, max as spark_max

In [0]:
gold_features = spark.table("workspace.census360.gold_gn_telco_features")

display(gold_features.limit(10))

In [0]:
stats = gold_features.select(
    spark_min("population_total").alias("min_population"),
    spark_max("population_total").alias("max_population"),
    spark_min("occupied_housing_units").alias("min_housing"),
    spark_max("occupied_housing_units").alias("max_housing"),
    spark_min("household_size").alias("min_household_size"),
    spark_max("household_size").alias("max_household_size"),
    spark_min("child_share").alias("min_child_share"),
    spark_max("child_share").alias("max_child_share"),
    spark_min("working_age_share").alias("min_working_age_share"),
    spark_max("working_age_share").alias("max_working_age_share")
).collect()[0]

stats

In [0]:
scored_gn_telco_features = gold_features.withColumn(
    "population_score",
    round(
        ((col("population_total") - stats["min_population"]) /
         (stats["max_population"] - stats["min_population"])) * 100,
        2
    )
).withColumn(
    "housing_score",
    round(
        ((col("occupied_housing_units") - stats["min_housing"]) /
         (stats["max_housing"] - stats["min_housing"])) * 100,
        2
    )
).withColumn(
    "household_size_score",
    round(
        ((col("household_size") - stats["min_household_size"]) /
         (stats["max_household_size"] - stats["min_household_size"])) * 100,
        2
    )
).withColumn(
    "child_share_score",
    round(
        ((col("child_share") - stats["min_child_share"]) /
         (stats["max_child_share"] - stats["min_child_share"])) * 100,
        2
    )
).withColumn(
    "working_age_score",
    round(
        ((col("working_age_share") - stats["min_working_age_share"]) /
         (stats["max_working_age_share"] - stats["min_working_age_share"])) * 100,
        2
    )
)

In [0]:
scored_gn_telco_features = scored_gn_telco_features.withColumn(
    "mobile_data_demand_score",
    round(
        0.50 * col("population_score") +
        0.35 * col("working_age_score") +
        0.15 * col("child_share_score"),
        2
    )
).withColumn(
    "home_broadband_opportunity_score",
    round(
        0.50 * col("housing_score") +
        0.30 * col("household_size_score") +
        0.20 * col("population_score"),
        2
    )
).withColumn(
    "youth_package_opportunity_score",
    round(
        0.45 * col("child_share_score") +
        0.35 * col("working_age_score") +
        0.20 * col("population_score"),
        2
    )
).withColumn(
    "rural_coverage_priority_score",
    round(
        0.45 * col("population_score") +
        0.35 * col("housing_score") +
        0.20 * col("household_size_score"),
        2
    )
).withColumn(
    "telco_demand_opportunity_score",
    round(
        0.35 * col("mobile_data_demand_score") +
        0.30 * col("home_broadband_opportunity_score") +
        0.20 * col("youth_package_opportunity_score") +
        0.15 * col("rural_coverage_priority_score"),
        2
    )
)

In [0]:
scored_gn_telco_features = scored_gn_telco_features.withColumn(
    "opportunity_tier",
    when(col("telco_demand_opportunity_score") >= 80, "Very High")
    .when(col("telco_demand_opportunity_score") >= 60, "High")
    .when(col("telco_demand_opportunity_score") >= 40, "Medium")
    .otherwise("Low")
)

In [0]:
scored_gn_telco_features.write.mode("overwrite").saveAsTable(
    "workspace.census360.gold_gn_telco_scored"
)

print("gold_gn_telco_scored created successfully.")

In [0]:
spark.table("workspace.census360.gold_gn_telco_scored").selectExpr(
    "count(*) as total_rows",
    "count(distinct gnd_uid) as distinct_gnd_uid",
    "min(telco_demand_opportunity_score) as min_score",
    "max(telco_demand_opportunity_score) as max_score"
).show()

In [0]:
spark.table("workspace.census360.gold_gn_telco_scored") \
    .groupBy("opportunity_tier") \
    .count() \
    .orderBy("opportunity_tier") \
    .show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when, percent_rank, round

In [0]:
scored_gn_telco_features = spark.table("workspace.census360.gold_gn_telco_scored")

In [0]:
window_spec = Window.orderBy(col("telco_demand_opportunity_score").asc())

scored_gn_telco_features_v2 = scored_gn_telco_features.withColumn(
    "opportunity_percentile",
    round(percent_rank().over(window_spec), 4)
)

In [0]:
scored_gn_telco_features_v2 = scored_gn_telco_features_v2.withColumn(
    "opportunity_tier",
    when(col("opportunity_percentile") >= 0.90, "Very High")
    .when(col("opportunity_percentile") >= 0.70, "High")
    .when(col("opportunity_percentile") >= 0.30, "Medium")
    .otherwise("Low")
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when, percent_rank, round

scored_gn_telco_features = spark.table("workspace.census360.gold_gn_telco_scored")

window_spec = Window.orderBy(col("telco_demand_opportunity_score").asc())

scored_gn_telco_features_v2 = scored_gn_telco_features.withColumn(
    "opportunity_percentile",
    round(percent_rank().over(window_spec), 4)
).withColumn(
    "opportunity_tier",
    when(col("opportunity_percentile") >= 0.90, "Very High")
    .when(col("opportunity_percentile") >= 0.70, "High")
    .when(col("opportunity_percentile") >= 0.30, "Medium")
    .otherwise("Low")
)

scored_gn_telco_features_v2.write.mode("overwrite").saveAsTable(
    "workspace.census360.gold_gn_telco_scored"
)

print("gold_gn_telco_scored updated with percentile-based tiers.")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when, percent_rank, round

# Load the existing scored table
scored_gn_telco_features = spark.table("workspace.census360.gold_gn_telco_scored")

# Create national percentile ranking
window_spec = Window.orderBy(col("telco_demand_opportunity_score").asc())

scored_gn_telco_features_v2 = scored_gn_telco_features.withColumn(
    "opportunity_percentile",
    round(percent_rank().over(window_spec), 4)
).withColumn(
    "opportunity_tier",
    when(col("opportunity_percentile") >= 0.90, "Very High")
    .when(col("opportunity_percentile") >= 0.70, "High")
    .when(col("opportunity_percentile") >= 0.30, "Medium")
    .otherwise("Low")
)

# Save as a NEW table first to avoid overwrite/read conflict
scored_gn_telco_features_v2.write.mode("overwrite").saveAsTable(
    "workspace.census360.gold_gn_telco_scored_v2"
)

print("gold_gn_telco_scored_v2 created successfully.")

In [0]:
spark.table("workspace.census360.gold_gn_telco_scored_v2").selectExpr(
    "count(*) as total_rows",
    "count(distinct gnd_uid) as distinct_gnd_uid",
    "min(telco_demand_opportunity_score) as min_score",
    "max(telco_demand_opportunity_score) as max_score",
    "min(opportunity_percentile) as min_percentile",
    "max(opportunity_percentile) as max_percentile"
).show()

In [0]:
spark.table("workspace.census360.gold_gn_telco_scored_v2") \
    .groupBy("opportunity_tier") \
    .count() \
    .orderBy("opportunity_tier") \
    .show()

In [0]:
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, round, max as spark_max
)

In [0]:
gold_scored = spark.table("workspace.census360.gold_gn_telco_scored_v2")

In [0]:
gold_district_telco_summary = gold_scored.groupBy(
    "province_code",
    "province_name",
    "district_code",
    "district_name"
).agg(
    count("gnd_uid").alias("gn_division_count"),
    spark_sum("population_total").alias("district_population_total"),
    spark_sum("occupied_housing_units").alias("district_housing_units"),
    round(avg("household_size"), 2).alias("avg_household_size"),
    round(avg("mobile_data_demand_score"), 2).alias("avg_mobile_data_demand_score"),
    round(avg("home_broadband_opportunity_score"), 2).alias("avg_home_broadband_opportunity_score"),
    round(avg("youth_package_opportunity_score"), 2).alias("avg_youth_package_opportunity_score"),
    round(avg("rural_coverage_priority_score"), 2).alias("avg_rural_coverage_priority_score"),
    round(avg("telco_demand_opportunity_score"), 2).alias("avg_telco_demand_opportunity_score"),
    spark_max("telco_demand_opportunity_score").alias("max_gn_opportunity_score")
)

In [0]:
gold_district_telco_summary = gold_district_telco_summary.withColumn(
    "district_opportunity_tier",
    when(col("avg_telco_demand_opportunity_score") >= 40, "Very High")
    .when(col("avg_telco_demand_opportunity_score") >= 25, "High")
    .when(col("avg_telco_demand_opportunity_score") >= 15, "Medium")
    .otherwise("Low")
)

In [0]:
gold_district_telco_summary.write.mode("overwrite").saveAsTable(
    "workspace.census360.gold_district_telco_summary"
)

print("gold_district_telco_summary created successfully.")

In [0]:
spark.table("workspace.census360.gold_district_telco_summary").selectExpr(
    "count(*) as district_rows",
    "count(distinct district_name) as district_count",
    "sum(district_population_total) as total_population",
    "sum(district_housing_units) as total_housing_units"
).show()

In [0]:
display(
    spark.table("workspace.census360.gold_district_telco_summary")
    .orderBy(col("avg_telco_demand_opportunity_score").desc())
)

In [0]:
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, round, max as spark_max, when
)

In [0]:
gold_scored = spark.table("workspace.census360.gold_gn_telco_scored_v2")

In [0]:
gold_ds_telco_summary = gold_scored.groupBy(
    "province_code",
    "province_name",
    "district_code",
    "district_name",
    "ds_division_code",
    "ds_division_name"
).agg(
    count("gnd_uid").alias("gn_division_count"),
    spark_sum("population_total").alias("ds_population_total"),
    spark_sum("occupied_housing_units").alias("ds_housing_units"),
    round(avg("household_size"), 2).alias("avg_household_size"),
    round(avg("mobile_data_demand_score"), 2).alias("avg_mobile_data_demand_score"),
    round(avg("home_broadband_opportunity_score"), 2).alias("avg_home_broadband_opportunity_score"),
    round(avg("youth_package_opportunity_score"), 2).alias("avg_youth_package_opportunity_score"),
    round(avg("rural_coverage_priority_score"), 2).alias("avg_rural_coverage_priority_score"),
    round(avg("telco_demand_opportunity_score"), 2).alias("avg_telco_demand_opportunity_score"),
    spark_max("telco_demand_opportunity_score").alias("max_gn_opportunity_score")
)

In [0]:
gold_ds_telco_summary = gold_ds_telco_summary.withColumn(
    "ds_opportunity_tier",
    when(col("avg_telco_demand_opportunity_score") >= 40, "Very High")
    .when(col("avg_telco_demand_opportunity_score") >= 25, "High")
    .when(col("avg_telco_demand_opportunity_score") >= 15, "Medium")
    .otherwise("Low")
)

In [0]:
gold_ds_telco_summary.write.mode("overwrite").saveAsTable(
    "workspace.census360.gold_ds_telco_summary"
)

print("gold_ds_telco_summary created successfully.")

In [0]:
spark.table("workspace.census360.gold_ds_telco_summary").selectExpr(
    "count(*) as ds_rows",
    "count(distinct ds_division_name) as ds_division_name_count",
    "sum(ds_population_total) as total_population",
    "sum(ds_housing_units) as total_housing_units"
).show()

In [0]:
display(
    spark.table("workspace.census360.gold_ds_telco_summary")
    .orderBy(col("avg_telco_demand_opportunity_score").desc())
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

In [0]:
gold_scored = spark.table("workspace.census360.gold_gn_telco_scored_v2")

In [0]:
ranking_window = Window.orderBy(col("telco_demand_opportunity_score").desc())

gold_telco_opportunity_ranking = gold_scored.withColumn(
    "national_opportunity_rank",
    row_number().over(ranking_window)
).select(
    "national_opportunity_rank",
    "province_name",
    "district_name",
    "ds_division_name",
    "gn_division_name",
    "gnd_uid",
    "population_total",
    "occupied_housing_units",
    "household_size",
    "mobile_data_demand_score",
    "home_broadband_opportunity_score",
    "youth_package_opportunity_score",
    "rural_coverage_priority_score",
    "telco_demand_opportunity_score",
    "opportunity_percentile",
    "opportunity_tier"
)

In [0]:
gold_telco_opportunity_ranking.write.mode("overwrite").saveAsTable(
    "workspace.census360.gold_telco_opportunity_ranking"
)

print("gold_telco_opportunity_ranking created successfully.")

In [0]:
spark.table("workspace.census360.gold_telco_opportunity_ranking").selectExpr(
    "count(*) as total_rows",
    "min(national_opportunity_rank) as min_rank",
    "max(national_opportunity_rank) as max_rank",
    "max(telco_demand_opportunity_score) as best_score"
).show()

In [0]:
display(
    spark.table("workspace.census360.gold_telco_opportunity_ranking")
    .orderBy(col("national_opportunity_rank").asc())
    .limit(20)
)

In [0]:
from pyspark.sql.functions import (
    countDistinct, count, sum as spark_sum, avg, round,
    when, col, lit
)

In [0]:
gold_scored = spark.table("workspace.census360.gold_gn_telco_scored_v2")

In [0]:
gold_dashboard_kpi_summary = gold_scored.agg(
    countDistinct("province_name").alias("total_provinces"),
    countDistinct("district_name").alias("total_districts"),
    countDistinct("ds_division_name").alias("total_ds_divisions"),
    countDistinct("gnd_uid").alias("total_gn_divisions"),
    spark_sum("population_total").alias("total_population"),
    spark_sum("occupied_housing_units").alias("total_housing_units"),
    round(avg("household_size"), 2).alias("avg_household_size"),
    round(avg("telco_demand_opportunity_score"), 2).alias("avg_telco_demand_opportunity_score"),
    spark_sum(when(col("opportunity_tier") == "Very High", 1).otherwise(0)).alias("very_high_opportunity_gns"),
    spark_sum(when(col("opportunity_tier") == "High", 1).otherwise(0)).alias("high_opportunity_gns"),
    spark_sum(when(col("opportunity_tier") == "Medium", 1).otherwise(0)).alias("medium_opportunity_gns"),
    spark_sum(when(col("opportunity_tier") == "Low", 1).otherwise(0)).alias("low_opportunity_gns")
).withColumn(
    "kpi_level", lit("National")
)

In [0]:
gold_dashboard_kpi_summary.write.mode("overwrite").saveAsTable(
    "workspace.census360.gold_dashboard_kpi_summary"
)

print("gold_dashboard_kpi_summary created successfully.")

In [0]:
display(
    spark.table("workspace.census360.gold_dashboard_kpi_summary")
)

In [0]:
spark.table("workspace.census360.gold_dashboard_kpi_summary").count()

In [0]:
gold_tables = [
    "workspace.census360.gold_gn_telco_features",
    "workspace.census360.gold_gn_telco_scored_v2",
    "workspace.census360.gold_district_telco_summary",
    "workspace.census360.gold_ds_telco_summary",
    "workspace.census360.gold_telco_opportunity_ranking",
    "workspace.census360.gold_dashboard_kpi_summary"
]

for table in gold_tables:
    row_count = spark.table(table).count()
    print(f"{table}: {row_count} rows")